In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import polars as pl

research_sandbox = Path(r'C:\\local\\GIT\\Public-Infrastructure-Service-Access\\Research-Sandbox')
sys.path.insert(0, str(research_sandbox / 'approximated_tradeoff' / 'src'))

from data_utilities import build_mapping
from mc_solvers import CreateIndexMapping, OptimizeWithGurobipy, PyomoOptimize


def load_thresholded_coverage(
    matrix_path,
    max_cover_dist_m=10_000,
    target_col='target_id',
    source_col='source_id',
    distance_col='total_dist',
):
    return (
        pl.scan_parquet(matrix_path)
        .filter(pl.col(distance_col) <= max_cover_dist_m)
        .select([target_col, source_col, distance_col])
        .unique()
        .collect()
        .to_pandas()
    )


def build_mc_solver_inputs(
    coverage,
    population_path,
    sources_path=None,
    fixed_source_type=None,
    target_col='target_id',
    source_col='source_id',
    population_id_col='ID',
    population_weight_col='population',
    source_id_col='ID',
):
    population = pd.read_parquet(
        population_path,
        columns=[population_id_col, population_weight_col],
    )
    population[population_id_col] = population[population_id_col].astype(str)
    target_ids = population[population_id_col].tolist()
    target_pos = {target_id: idx for idx, target_id in enumerate(target_ids)}
    weights = population[population_weight_col].to_numpy(dtype=float)

    source_ids = sorted(coverage[source_col].astype(str).unique())
    fixed_source_ids = set()
    if sources_path is not None:
        sources = pd.read_parquet(sources_path, columns=[source_id_col, 'source_type'])
        sources[source_id_col] = sources[source_id_col].astype(str)
        source_ids = sources[source_id_col].tolist()
        if fixed_source_type is not None:
            fixed_source_ids = set(
                sources.loc[
                    sources['source_type'].astype(str) == fixed_source_type,
                    source_id_col,
                ]
            )

    source_pos = {source_id: idx for idx, source_id in enumerate(source_ids)}
    pairs = coverage.copy()
    pairs[target_col] = pairs[target_col].astype(str)
    pairs[source_col] = pairs[source_col].astype(str)
    pairs = pairs[
        pairs[target_col].isin(target_pos)
        & pairs[source_col].isin(source_pos)
    ]
    pairs['i'] = pairs[target_col].map(target_pos).astype(np.int64)
    pairs['j'] = pairs[source_col].map(source_pos).astype(np.int64)

    all_facs = build_mapping(
        pairs,
        threshold=max(float(pairs['total_dist'].max()), 0.0),
        key_col='j',
        value_col='i',
    )
    mapping = CreateIndexMapping(all_facs, target_ids)
    fixed_j = [source_pos[source_id] for source_id in fixed_source_ids if source_id in source_pos]
    fixed_j = sorted(set(fixed_j).intersection(set(mapping.J.tolist())))

    return {
        'weights': weights,
        'mapping': mapping,
        'target_ids': target_ids,
        'source_ids': source_ids,
        'fixed_j': fixed_j,
        'coverage_pairs': pairs,
    }


def summarize_solution(result_row, source_ids, fixed_j=None):
    fixed_j = set() if fixed_j is None else set(fixed_j)
    solution = result_row['solution'] or []
    selected_new = [j for j in solution if j not in fixed_j]
    return {
        'termination': result_row['termination'],
        'covered_population': result_row['value'],
        'solver_time_s': result_row['solving'],
        'upper_bound': result_row['upper'],
        'selected_source_ids': [source_ids[j] for j in solution],
        'selected_new_source_ids': [source_ids[j] for j in selected_new],
    }


In [ ]:
data = Path(r'C:\\local\\Download_Depot\\east-timor_data\\outputs')
run_tag = 'pop_1_sample_1_max_none_agg_8_maxdist_10000_amenity_all_candidates_spacing_5000_maxsnap_5000'

matrix_path = data / f'distance_matrix_{run_tag}.parquet'
population_path = data / f'population_{run_tag}.parquet'
sources_path = data / f'sources_{run_tag}.parquet'

max_cover_dist_m = 10_000
p_facilities = 20

coverage = load_thresholded_coverage(matrix_path, max_cover_dist_m=max_cover_dist_m)
inputs = build_mc_solver_inputs(
    coverage=coverage,
    population_path=population_path,
    sources_path=sources_path,
)

mapping = inputs['mapping']
solution_summary = OptimizeWithGurobipy(
    w=inputs['weights'],
    I=mapping.I,
    J=mapping.J,
    IJ=mapping.IJ,
    budget_list=[p_facilities],
    maxTimeInSeconds=300,
    mipGap=1e-6,
    trace=False,
)

solution_details = summarize_solution(
    solution_summary.loc[p_facilities],
    source_ids=inputs['source_ids'],
)
solution_summary.drop(columns=['solution'])


In [ ]:
vietnam_data = Path(r'C:\\local\\Download_Depot\\vietnam_data\\outputs')
vietnam_tag = 'pop_1_sample_1_max_none_agg_10_maxdist_150000_amenity_all_candidates_spacing_10000_maxsnap_5000'

vietnam_matrix_path = vietnam_data / f'distance_matrix_{vietnam_tag}.parquet'
vietnam_population_path = vietnam_data / f'population_{vietnam_tag}.parquet'
vietnam_sources_path = vietnam_data / f'sources_{vietnam_tag}.parquet'

vietnam_coverage = load_thresholded_coverage(
    vietnam_matrix_path,
    max_cover_dist_m=1_000,
)
vietnam_inputs = build_mc_solver_inputs(
    coverage=vietnam_coverage,
    population_path=vietnam_population_path,
    sources_path=vietnam_sources_path,
    fixed_source_type='existing',
)

vietnam_mapping = vietnam_inputs['mapping']
vietnam_budget = len(vietnam_inputs['fixed_j']) + 100
vietnam_solution = OptimizeWithGurobipy(
    w=vietnam_inputs['weights'],
    I=vietnam_mapping.I,
    J=vietnam_mapping.J,
    IJ=vietnam_mapping.IJ,
    budget_list=[vietnam_budget],
    maxTimeInSeconds=3600,
    mipGap=1e-6,
    trace=False,
    already_open=vietnam_inputs['fixed_j'],
)

vietnam_solution_details = summarize_solution(
    vietnam_solution.loc[vietnam_budget],
    source_ids=vietnam_inputs['source_ids'],
    fixed_j=vietnam_inputs['fixed_j'],
)
vietnam_solution.drop(columns=['solution'])


In [ ]:
from distance_pipeline.viz import build_solution_layers, plot_max_cover_solution


In [ ]:
base_path_for_document = Path(r'C:\\Users\\joaqu\\Dropbox\\Apps\\Overleaf\\Real Life Distance Generator')
figures_path = base_path_for_document / 'figures'
optimization_figure_path = figures_path / 'timor_leste_max_cover_solution.png'

class SolverResultAdapter:
    def __init__(self, solution):
        self.J = solution['selected_source_ids']
        self.x = {source_id: type('Value', (), {'value': 1.0})() for source_id in self.J}

model_for_plot = SolverResultAdapter(solution_details)
solution_layers = build_solution_layers(
    model=model_for_plot,
    matrix_path=matrix_path,
    matrix=coverage,
    population_path=population_path,
    sources_path=sources_path,
    max_cover_dist_m=max_cover_dist_m,
)

fig, ax = plot_max_cover_solution(
    solution_layers,
    title='Timor-Leste maximum covering solution, 10 km threshold',
    output_path=optimization_figure_path,
)
